# Dynamic, per-user MCP servers — runnable prototype

This notebook prototypes the design for **per-client custom MCP server URLs** on top of the
Relivo agent, *without any database changes* — everything is in-memory so you can run and
tweak sub-parts freely.

It mirrors the real code paths:

| Prototype piece | Maps to real file |
| --- | --- |
| `InMemoryMcpStore` | future `user_mcp_servers` table + `services/mcp_server_service.py` |
| `validate_mcp_url` | SSRF guard in the CRUD/service layer |
| `McpToolRegistry` | generalized `src/tools/firecrawl_mcp.py` |
| `build_agent` / `BaseAgentLike` | `src/agents/base_agent.py` |
| `AgentRegistry` | new agent-graph cache in `src/agents/orchestrator.py` |
| `resolve_agent_for_user` | wiring inside `src/services/chat_service.py` |

**The core idea:** today the agent is a process-wide singleton with a *fixed* tool list.
To let each client bring their own remote MCP tools, we resolve a **per-user tool set** at
request time and use **two caches** so it stays cheap:

1. **Tool cache** — keyed by a hash of `(url, transport, auth)`; only hits the network on a miss.
2. **Agent-graph cache** — LRU keyed by the sorted set of a user's config hashes. A user with
   no custom servers falls straight through to the shared base agent (zero overhead).

> Each section is runnable on its own after you run **Section 0 (imports)**. The full demo in
> Section 7 runs **offline** using a mock MCP loader; Section 8 shows the real
> `MultiServerMCPClient` call you'd swap in.

## Section 0 — Imports & setup

Run this first. Only uses libraries already in the project's `.venv`.

In [ ]:
from __future__ import annotations

import asyncio
import hashlib
import ipaddress
import logging
import os
import socket
import time
from collections import OrderedDict
from collections.abc import Awaitable, Callable, Sequence
from dataclasses import dataclass
from typing import Any
from urllib.parse import urlparse

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("mcp_prototype")

# LangChain tool primitive (already installed for the project).
from langchain_core.tools import BaseTool, tool

print("imports ok")

## Section 1 — Per-user MCP config store (in-memory, replaces the DB)

In production this is the `user_mcp_servers` table. Here it's a dict `user_id -> [configs]`.

`config_hash()` is the stable cache key. It intentionally includes the auth secret so that
rotating a token invalidates the cache automatically.

In [ ]:
@dataclass(frozen=True, slots=True)
class McpServerConfig:
    """One remote MCP server a user has registered (remote URL only — no local process)."""

    user_id: str
    name: str
    url: str
    transport: str = "http"          # "http" | "sse"
    auth_type: str = "none"          # "none" | "bearer" | "header"
    auth_secret: str = ""            # encrypted at rest in prod; plain here for the prototype
    auth_header: str = "Authorization"
    enabled: bool = True

    def config_hash(self) -> str:
        """Stable key for the tool cache. Rotating the secret changes the hash."""
        raw = "|".join(
            [self.url, self.transport, self.auth_type, self.auth_header, self.auth_secret]
        )
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]

    def client_config(self) -> dict[str, Any]:
        """Translate to a langchain-mcp-adapters MultiServerMCPClient server entry."""
        entry: dict[str, Any] = {"transport": self.transport, "url": self.url}
        if self.auth_type == "bearer" and self.auth_secret:
            entry["headers"] = {"Authorization": f"Bearer {self.auth_secret}"}
        elif self.auth_type == "header" and self.auth_secret:
            entry["headers"] = {self.auth_header: self.auth_secret}
        return entry


class InMemoryMcpStore:
    """Stand-in for the user_mcp_servers table + CRUD service."""

    def __init__(self) -> None:
        self._by_user: dict[str, list[McpServerConfig]] = {}

    def add(self, config: McpServerConfig) -> None:
        servers = self._by_user.setdefault(config.user_id, [])
        # unique(user_id, name)
        servers[:] = [s for s in servers if s.name != config.name]
        servers.append(config)

    def remove(self, user_id: str, name: str) -> None:
        self._by_user[user_id] = [
            s for s in self._by_user.get(user_id, []) if s.name != name
        ]

    def list_enabled(self, user_id: str) -> list[McpServerConfig]:
        return [s for s in self._by_user.get(user_id, []) if s.enabled]


store = InMemoryMcpStore()
print("store ready")

## Section 2 — SSRF guard

Users supply arbitrary URLs, so this is the **highest-risk** piece. Before we ever connect we:

- require `https` (a dev override allows `http`),
- reject `localhost` and other blocked names,
- resolve DNS and reject **every** resolved IP that is loopback / private / link-local /
  reserved / multicast — this blocks cloud metadata endpoints like `169.254.169.254`.

The returned IP list is what you'd **pin** for the actual connection to defeat DNS-rebinding
(resolve once, validate, then connect to that exact IP).

In [ ]:
class McpUrlError(ValueError):
    """Raised when a user-supplied MCP URL fails validation."""


BLOCKED_HOSTNAMES = {"localhost", "localhost.localdomain", "ip6-localhost"}


def _resolve_ips(host: str) -> list[str]:
    """Return resolved IPs. IP literals skip DNS so the demo runs offline."""
    try:
        ipaddress.ip_address(host)
        return [host]
    except ValueError:
        pass
    infos = socket.getaddrinfo(host, None)
    return sorted({info[4][0] for info in infos})


def _is_blocked_ip(ip: str) -> bool:
    addr = ipaddress.ip_address(ip)
    return (
        addr.is_private
        or addr.is_loopback
        or addr.is_link_local      # covers 169.254.169.254 metadata endpoint
        or addr.is_reserved
        or addr.is_multicast
        or addr.is_unspecified
    )


def validate_mcp_url(
    url: str,
    *,
    allow_http: bool = False,
    allowlist: set[str] | None = None,
) -> list[str]:
    """Validate a user MCP URL and return the resolved IPs to pin for the connection."""
    parsed = urlparse(url)
    if parsed.scheme == "https" or (allow_http and parsed.scheme == "http"):
        pass
    else:
        raise McpUrlError(f"scheme not allowed: {parsed.scheme!r} (https required)")

    host = parsed.hostname
    if not host:
        raise McpUrlError("missing host")
    if host.lower() in BLOCKED_HOSTNAMES:
        raise McpUrlError(f"blocked host: {host}")
    if allowlist is not None and host not in allowlist:
        raise McpUrlError(f"host not in allowlist: {host}")

    ips = _resolve_ips(host)
    for ip in ips:
        if _is_blocked_ip(ip):
            raise McpUrlError(f"blocked internal IP {ip} for host {host}")
    return ips


# --- quick offline checks (no DNS needed for these) ---
def _expect_reject(url: str, **kw: Any) -> None:
    try:
        validate_mcp_url(url, **kw)
    except McpUrlError as exc:
        print(f"REJECTED  {url:45s} -> {exc}")
    else:
        print(f"!! ALLOWED (unexpected) {url}")


_expect_reject("http://mcp.example.com/v2/mcp")          # not https
_expect_reject("https://localhost:8000/mcp")             # blocked name
_expect_reject("https://169.254.169.254/latest/meta")    # cloud metadata
_expect_reject("https://10.0.0.5/mcp")                   # private range
print("OK        https://1.1.1.1/mcp  ->", validate_mcp_url("https://1.1.1.1/mcp"))

## Section 3 — MCP tool registry (TTL cache keyed by config hash)

This generalizes `src/tools/firecrawl_mcp.py`. The registry takes a *pluggable loader* so we
can run fully offline with a **mock loader**, then swap in the real
`MultiServerMCPClient` loader (Section 8) with no other changes.

Properties it demonstrates:
- **TTL cache** per config hash → misses hit the network, hits are free.
- **Fault isolation** — one dead server is logged and skipped; others still load.
- **Namespacing** — tool names are prefixed `mcp__<server>__…` to avoid collisions.
- **Injectable clock** so we can force expiry in the demo.

In [ ]:
ToolLoader = Callable[[McpServerConfig], Awaitable[list[BaseTool]]]


def _namespaced(server_name: str, base: BaseTool) -> BaseTool:
    """Prefix a tool name so two servers can't collide."""
    prefixed = f"mcp__{server_name}__{base.name}"
    # StructuredTool lets us clone with a new name cheaply for the prototype.
    return base.model_copy(update={"name": prefixed})


class McpToolRegistry:
    """Per-config TTL cache over an MCP tool loader."""

    def __init__(
        self,
        loader: ToolLoader,
        *,
        ttl_seconds: float = 300.0,
        clock: Callable[[], float] = time.monotonic,
        max_tools_per_user: int = 64,
    ) -> None:
        self._loader = loader
        self._ttl = ttl_seconds
        self._clock = clock
        self._max_tools = max_tools_per_user
        self._cache: dict[str, tuple[float, list[BaseTool]]] = {}
        self.stats = {"hits": 0, "misses": 0}

    def invalidate(self, config_hash: str) -> None:
        self._cache.pop(config_hash, None)

    async def _get_one(self, config: McpServerConfig) -> list[BaseTool]:
        key = config.config_hash()
        now = self._clock()
        cached = self._cache.get(key)
        if cached and cached[0] > now:
            self.stats["hits"] += 1
            return cached[1]

        self.stats["misses"] += 1
        raw = await self._loader(config)
        tools = [_namespaced(config.name, t) for t in raw]
        self._cache[key] = (now + self._ttl, tools)
        return tools

    async def _safe_get_one(self, config: McpServerConfig) -> list[BaseTool]:
        try:
            return await self._get_one(config)
        except Exception as exc:  # fault isolation: never let one server break the turn
            logger.warning("MCP load failed server=%s: %s", config.name, exc)
            return []

    async def get_tools_for_configs(
        self, configs: Sequence[McpServerConfig]
    ) -> list[BaseTool]:
        groups = await asyncio.gather(*(self._safe_get_one(c) for c in configs))
        tools = [t for group in groups for t in group]
        if len(tools) > self._max_tools:
            logger.warning("Capping MCP tools %s -> %s", len(tools), self._max_tools)
            tools = tools[: self._max_tools]
        return tools


print("registry defined")

### 3a — A mock MCP loader (so the notebook runs offline)

Each "server" returns real, callable LangChain tools. This stands in for the network round
trip that `MultiServerMCPClient.get_tools()` performs against a remote MCP URL. It also counts
calls so we can *see* the cache working.

In [ ]:
MOCK_LOAD_CALLS: dict[str, int] = {}


def _make_mock_tools(server_name: str) -> list[BaseTool]:
    if server_name == "weather":
        @tool
        def get_forecast(city: str) -> str:
            """Return a (fake) weather forecast for a city."""
            return f"{city}: 24C and sunny (mock)"

        @tool
        def severe_alerts(region: str) -> str:
            """Return (fake) severe-weather alerts for a region."""
            return f"No active alerts for {region} (mock)"

        return [get_forecast, severe_alerts]

    if server_name == "jira":
        @tool
        def create_issue(project: str, title: str) -> str:
            """Create a (fake) Jira issue and return its key."""
            return f"{project}-123 created: {title!r} (mock)"

        return [create_issue]

    @tool
    def ping() -> str:
        """Health ping for an unknown mock server."""
        return "pong (mock)"

    return [ping]


async def mock_loader(config: McpServerConfig) -> list[BaseTool]:
    """Pretend to connect to a remote MCP URL and list its tools."""
    # In prod, validate_mcp_url(config.url) would gate this before connecting.
    MOCK_LOAD_CALLS[config.name] = MOCK_LOAD_CALLS.get(config.name, 0) + 1
    await asyncio.sleep(0.05)  # simulate network latency
    if config.name == "broken":
        raise ConnectionError("connection refused (simulated)")
    return _make_mock_tools(config.name)


print("mock loader ready")

## Section 4 — Base tools + a `BaseAgent`-like wrapper

This mirrors `src/agents/base_agent.py`: `create_agent(...)` binds tools **at build time**.
Without an `OPENAI_API_KEY` we fall back to a `FakeListChatModel` exactly like the project's
orchestrator does, so the notebook builds an agent either way.

The architectural point we want to show is simply: **the bound tool set differs per user.**
If you export a real `OPENAI_API_KEY`, Section 7 will actually invoke a turn and you'll see the
model call an MCP tool.

In [ ]:
import warnings

from langchain_core._api.deprecation import LangChainPendingDeprecationWarning

warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)

from langchain.agents import create_agent
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langgraph.checkpoint.memory import InMemorySaver


# --- built-in tools every agent gets (stand-in for memory_* / read_chat_attachment) ---
@tool
def get_demo_context() -> str:
    """Return demo context available to every user."""
    return "demo-context: base tools are always available"


BASE_TOOLS: list[BaseTool] = [get_demo_context]


def _build_model() -> Any:
    if os.getenv("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(
            model=os.getenv("RELIVO_CHAT_MODEL", "gpt-5-mini"),
            reasoning_effort=os.getenv("RELIVO_CHAT_REASONING_EFFORT", "low"),
            use_responses_api=True,
        )
    logger.info("No OPENAI_API_KEY: using FakeListChatModel (tool set still differs per user)")
    return FakeListChatModel(responses=["(demo fallback) set OPENAI_API_KEY to run real turns"])


@dataclass(slots=True)
class BaseAgentLike:
    """Tiny stand-in for src.agents.base_agent.BaseAgent."""

    name: str
    tools: list[BaseTool]
    graph: Any

    @property
    def tool_names(self) -> list[str]:
        return [t.name for t in self.tools]


def build_agent(tools: list[BaseTool], *, name: str = "Orchestrator") -> BaseAgentLike:
    """Build a LangGraph agent with the given tools bound in (like create_agent in the project)."""
    graph = create_agent(
        model=_build_model(),
        tools=tools,
        system_prompt="You are the Relivo Orchestrator. Use tools when they help.",
        checkpointer=InMemorySaver(),
        name=name,
    )
    return BaseAgentLike(name=name, tools=tools, graph=graph)


print("agent builder ready")

## Section 5 — Agent registry (signature cache)

We must **not** mutate the shared base graph per request (that would leak one client's tools
into another's queries). Instead we cache one built graph per **tool signature** — the sorted
set of a user's config hashes plus a base version. Same signature ⇒ same cached agent.

In [ ]:
BASE_VERSION = "base@v1"


def agent_signature(configs: Sequence[McpServerConfig]) -> tuple[str, ...]:
    """Stable signature: base version + sorted config hashes."""
    return (BASE_VERSION, *sorted(c.config_hash() for c in configs))


class AgentRegistry:
    """LRU cache of built agents keyed by tool signature."""

    def __init__(
        self,
        base_tools: list[BaseTool],
        builder: Callable[[list[BaseTool]], BaseAgentLike],
        *,
        max_size: int = 32,
    ) -> None:
        self._base = base_tools
        self._builder = builder
        self._max = max_size
        self._cache: OrderedDict[tuple[str, ...], BaseAgentLike] = OrderedDict()
        self.stats = {"hits": 0, "misses": 0}

    def resolve(
        self, signature: tuple[str, ...], mcp_tools: list[BaseTool]
    ) -> BaseAgentLike:
        if signature in self._cache:
            self.stats["hits"] += 1
            self._cache.move_to_end(signature)
            return self._cache[signature]

        self.stats["misses"] += 1
        agent = self._builder(self._base + mcp_tools)
        self._cache[signature] = agent
        self._cache.move_to_end(signature)
        if len(self._cache) > self._max:
            evicted, _ = self._cache.popitem(last=False)
            logger.info("Evicted agent signature=%s", evicted)
        return agent


print("AgentRegistry ready")

## Section 6 — Chat-flow wiring

This is the glue that would live inside `ChatService.stream_chat`, right after it resolves
`user_id` from the conversation. A user with **no** custom servers gets `signature == (BASE_VERSION,)`
and shares one cached base agent — identical to today's singleton behaviour.

In [ ]:
# Wire the pieces together.
tool_registry = McpToolRegistry(mock_loader, ttl_seconds=300.0)
agent_registry = AgentRegistry(BASE_TOOLS, build_agent)


async def resolve_agent_for_user(user_id: str) -> BaseAgentLike:
    """Resolve the per-user agent: base tools + the user's enabled MCP tools (cached)."""
    configs = store.list_enabled(user_id)
    signature = agent_signature(configs)
    mcp_tools = await tool_registry.get_tools_for_configs(configs)
    return agent_registry.resolve(signature, mcp_tools)


print("wiring ready")

## Section 7 — End-to-end demo (offline)

- **user_b** registers nothing → gets only base tools.
- **user_a** registers `weather` + `jira` + a `broken` server → gets base + namespaced MCP
  tools, and the broken one is skipped without failing the turn.
- We resolve **user_a twice** to show both caches serving hits.
- We register a new server for user_a to show the signature (and thus the agent) changing.

In [ ]:
async def demo() -> None:
    # user_b: no custom MCP servers
    agent_b = await resolve_agent_for_user("user_b")
    print("user_b tools:", agent_b.tool_names)

    # user_a: three servers, one deliberately broken
    store.add(McpServerConfig(user_id="user_a", name="weather",
                              url="https://mcp.weather.example/v2/mcp",
                              auth_type="bearer", auth_secret="tok-123"))
    store.add(McpServerConfig(user_id="user_a", name="jira",
                              url="https://mcp.jira.example/mcp"))
    store.add(McpServerConfig(user_id="user_a", name="broken",
                              url="https://mcp.broken.example/mcp"))

    agent_a1 = await resolve_agent_for_user("user_a")
    print("user_a tools:", agent_a1.tool_names)

    # resolve again -> both caches should hit, no new mock loads
    agent_a2 = await resolve_agent_for_user("user_a")
    print("same agent instance reused:", agent_a1 is agent_a2)

    # add a server -> signature changes -> new agent built
    store.add(McpServerConfig(user_id="user_a", name="notion",
                              url="https://mcp.notion.example/mcp"))
    agent_a3 = await resolve_agent_for_user("user_a")
    print("agent rebuilt after adding server:", agent_a3 is not agent_a1)
    print("user_a tools now:", agent_a3.tool_names)

    print()
    print("mock network loads per server:", MOCK_LOAD_CALLS)
    print("tool cache stats:", tool_registry.stats)
    print("agent cache stats:", agent_registry.stats)


await demo()

### 7a — TTL expiry forces a fresh load

Swap in a manual clock to prove expired entries re-fetch while unexpired ones stay cached.

In [ ]:
class FakeClock:
    def __init__(self) -> None:
        self.t = 1000.0

    def __call__(self) -> float:
        return self.t


clock = FakeClock()
ttl_registry = McpToolRegistry(mock_loader, ttl_seconds=60.0, clock=clock)
cfg = McpServerConfig(user_id="u", name="weather", url="https://mcp.weather.example/mcp")

MOCK_LOAD_CALLS.clear()
await ttl_registry.get_tools_for_configs([cfg])   # miss -> load
await ttl_registry.get_tools_for_configs([cfg])   # hit  -> no load
clock.t += 61                                      # advance past TTL
await ttl_registry.get_tools_for_configs([cfg])   # expired -> reload

print("loads for 'weather':", MOCK_LOAD_CALLS.get("weather"), "(expected 2)")
print("ttl registry stats:", ttl_registry.stats)

### 7b — Optional: run a real turn (needs `OPENAI_API_KEY`)

If a real key is set, this drives `agent_a3` and prints the tool calls the model makes. With
the fake model it just prints the fallback text — the tool *set* difference above is the point.

In [ ]:
async def run_turn(agent: BaseAgentLike, message: str, thread_id: str = "demo") -> None:
    result = await agent.graph.ainvoke(
        {"messages": [{"role": "user", "content": message}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    for m in result["messages"]:
        calls = getattr(m, "tool_calls", None)
        if calls:
            print("tool_calls:", [(c["name"], c["args"]) for c in calls])
    print("final:", result["messages"][-1].content)


if os.getenv("OPENAI_API_KEY"):
    agent_a = await resolve_agent_for_user("user_a")
    await run_turn(
        agent_a, "What's the weather in Pune and file a Jira issue in OPS titled 'demo'?"
    )
else:
    # FakeListChatModel can't bind tools at invocation time, so skip the live turn.
    print("Skipping live turn: set OPENAI_API_KEY to see the model call an MCP tool.")

## Section 8 — The real `MultiServerMCPClient` loader

Everything above used `mock_loader`. To go live against real remote MCP URLs, swap the loader
for this one — it runs the SSRF guard, then lists tools over HTTP exactly like
`src/tools/firecrawl_mcp.py` does today. **No other code changes**: the registry, agent
resolution, and chat wiring are untouched.

Set `USE_REAL_MCP = True` and provide a reachable MCP URL to try it.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

USE_REAL_MCP = False  # flip to True and edit the config below to test a real server


async def real_loader(config: McpServerConfig) -> list[BaseTool]:
    """Production loader: validate the URL, then discover tools over HTTP."""
    validate_mcp_url(config.url, allow_http=False)  # SSRF guard BEFORE any connection
    client = MultiServerMCPClient({config.name: config.client_config()})
    tools = await client.get_tools(server_name=config.name)
    logger.info("Loaded %s tools from MCP server=%s", len(tools), config.name)
    return tools


if USE_REAL_MCP:
    real_registry = McpToolRegistry(real_loader, ttl_seconds=300.0)
    real_agent_registry = AgentRegistry(BASE_TOOLS, build_agent)

    store.add(
        McpServerConfig(
            user_id="real_user",
            name="firecrawl",
            url="https://mcp.firecrawl.dev/v2/mcp",
            auth_type="bearer",
            auth_secret=os.getenv("FIRECRAWL_API_KEY", ""),
        )
    )
    cfgs = store.list_enabled("real_user")
    tools = await real_registry.get_tools_for_configs(cfgs)
    print("real tools:", [t.name for t in tools])
else:
    print("USE_REAL_MCP is False — using the offline mock loader above.")

## Recap — how this answers the original question

- **"Agent is loaded once at server start and answers everyone."** → Keep that base agent, but
  resolve a **per-user** agent from the `AgentRegistry`. No-custom-server users share the base
  agent (unchanged behaviour, zero overhead).
- **"A client adds an MCP URL — how does the agent use those tools?"** → The URL is stored per
  user (`InMemoryMcpStore` → `user_mcp_servers` table). At request time
  `resolve_agent_for_user` loads that server's tools (cached) and builds/reuses an agent with
  them **bound in**, isolated from other clients.
- **"Remote MCP only, no local run."** → `transport="http"` + `MultiServerMCPClient` in
  `real_loader`; cached tool objects reconnect per call, so nothing runs locally.
- **Cost stays low** thanks to the two caches; **safety** comes from `validate_mcp_url` (SSRF)
  running before any connection.

To productionize, replace `InMemoryMcpStore` with the `user_mcp_servers` table + CRUD API,
move `validate_mcp_url` into that service, generalize `firecrawl_mcp.py` into `McpToolRegistry`,
and call `resolve_agent_for_user` inside `ChatService.stream_chat`.